# 💧 LFM2 - ORPO with MLX

This tutorial demonstrates how to fine-tune our LFM2 models, e.g. [`LiquidAI/LFM2.5-1.2B-Instruct`](https://huggingface.co/LiquidAI/LFM2.5-1.2B-Instruct), using the MLX-LM-LoRA library.

Follow along if it's your first time using MLX, or take single code snippets for your own workflow

## 🎯 What You'll Find:
- **ORPO** - Align with human preferences  

## 📋 Prerequisites:
- **M-Series Mac**: Any Mac with a M-Series and minimum 24GB of RAM
- **Hugging Face Account**: For accessing models and datasets



# 📦 Installation & Setup

First, let's install all the required packages:


In [ ]:
%%capture
!pip install mlx-lm-lora

Let's now import the packages are installed correctly

In [3]:
from mlx_lm_lora.utils import from_pretrained, save_pretrained_merged, calculate_iters
from mlx_lm_lora.trainer.orpo_trainer import ORPOTrainingArgs, train_orpo
from mlx_lm_lora.trainer.datasets import CacheDataset, PreferenceDataset
from datasets import load_dataset

from mlx_lm.tuner.utils import print_trainable_parameters, build_schedule

import mlx.optimizers as optim

# Loading the model from MLX-LM-LoRA



In [4]:
# Select a model to fine-tune from the list
lfm_models = [
    "LiquidAI/LFM2.5-8B-A1B",
    "LiquidAI/LFM2.5-350M",
    "LiquidAI/LFM2.5-1.2B-Thinking",
    "LiquidAI/LFM2.5-1.2B-Instruct",
    "LiquidAI/LFM2.5-1.2B-JP-202606",
    "LiquidAI/LFM2-2.6B-Exp",
    "LiquidAI/LFM2-2.6B",
    "LiquidAI/LFM2-700M",
    "LiquidAI/LFM2-350M",
]

# Model to fine-tune
model_id = "LiquidAI/LFM2.5-1.2B-Instruct"
new_model_name = "lfm2-orpo"

# LoRA adapter configuration
lora_config = {
    "rank": 12,  # Low-rank bottleneck size (Larger rank = smarter, but slower). Suggested 8, 16, 32, 64, 128
    "dropout": 0.0,
    "scale": 10.0, # Multiplier for how hard the LoRA update hits the base weights
    "use_dora": False,
    "num_layers": 12 # Use -1 for all layers
}

# Quantization configuration
quantized_config = {
    "bits": 4,
    "group_size": 64,
    "mode": "affine",
}

# Load the model and tokenizer
model, tokenizer, adapter_file = from_pretrained(
    model=model_id,
    lora_config=lora_config,
    quantized_load=quantized_config,
    new_adapter_path=f"./{new_model_name}"
)
print_trainable_parameters(model)

# 🎯 Odds Ratio Preference Optimization (ORPO + LoRA)

ORPO aligns the model with human preferences by learning from preference pairs (chosen vs. rejected responses). This typically follows SFT training.

ORPO is a more efficient alternative to DPO since it doesn’t require a separate reference model. Here, we use LoRA (Low-Rank Adaptation) to fine-tune the model by training only a small number of additional parameters. Perfect for limited compute resources!

## Load a DPO Dataset

We will use [mlabonne/orpo-dpo-mix-40k](https://huggingface.co/datasets/mlabonne/orpo-dpo-mix-40k), limiting ourselves to the first 1k samples for brevity. Feel free to change the limit by changing the slicing index in the parameter `split`. The size of the validation data can be adjusted by changing `test_size`.

In [ ]:
def format(sample):
    sample["chosen"] = tokenizer.apply_chat_template(
        conversation=sample["chosen"],
        add_generation_prompt=False,
        tokenize=False
    )

    sample["rejected"] = tokenizer.apply_chat_template(
        conversation=sample["rejected"],
        add_generation_prompt=False,
        tokenize=False
    )
    return sample

print("📥 Loading DPO/ORPO dataset...")
dataset_orpo = load_dataset("mlabonne/orpo-dpo-mix-40k", split="train[:1000]").map(format)
dataset_orpo = dataset_orpo.train_test_split(test_size=0.1, seed=42)
train_dataset_orpo, eval_dataset_orpo = dataset_orpo['train'], dataset_orpo['test']

train_set = PreferenceDataset(train_dataset_orpo, tokenizer, chosen_key="chosen", rejected_key="rejected")
eval_set = PreferenceDataset(eval_dataset_orpo, tokenizer, chosen_key="chosen", rejected_key="rejected")

print("✅ OPRO Dataset loaded:")
print(f"   📚 Train samples: {len(train_dataset_orpo)}")
print(f"   🧪 Eval samples: {len(eval_dataset_orpo)}")

sample = train_dataset_orpo[0]
print("\n📝 Single Sample:")
print(f"   ✅ Chosen: {sample['chosen'][:1000]}...")
print(f"   ❌ Rejected: {sample['rejected'][:1000]}...")

## Launch Training

We are now ready to launch a ORPO run with `train_orpo`, feel free to modify `ORPOTrainingArgs` to play around with different configurations.

In [ ]:
batch_size = 1 # Number of training samples processed in each forward/backward pass. A batch size of 1 minimizes memory usage.
epochs = 1 # Number of complete passes through the training dataset.

# Convert the desired number of epochs into the number of training iterations.
iters = calculate_iters(
    train_set,
    batch_size=batch_size,
    epochs=epochs
)

lr = build_schedule(
    schedule_config={
        "name": "cosine_decay", # Cosine decay gradually reduces the learning rate during training.
        "warmup": 40, # Number of initial steps used to gradually increase the learning rate. Warmup helps avoid unstable updates at the beginning of training.
        "warmup_init": 2e-7, # Learning rate used at the very beginning of the warmup period.
        "arguments": [
            2e-5,               # Peak learning rate after warmup
            int(iters * 0.95),  # Decay LR over roughly 95% of training
            2e-6                # Final/minimum learning rate
        ],
    }
)

opt = optim.AdamW(
    learning_rate=lr, # Uses the dynamic learning-rate schedule defined above.
    betas=[0.9, 0.999], # Adam momentum coefficients. beta1 controls the moving average of gradients. beta2 controls the moving average of squared gradients.
    eps=1e-8, # Small numerical-stability constant used by AdamW.
    weight_decay=0.00, # Strength of weight decay regularization. 0.00 disables weight decay.
    bias_correction=False # Disables Adam's bias correction for the moving averages.
)

print("🏗️  Creating ORPO trainer configuration...")
orpo_config = ORPOTrainingArgs(
    batch_size=batch_size,
    iters=iters,
    beta=0.1, # Controls the strength of the preference optimization objective relative to the SFT objective.
    reward_scaling=1.0, # Scales the preference/reward signal; higher values increase the magnitude of preference differences.
    gradient_accumulation_steps=4, # Accumulate gradients across 4 micro-batches before updating weights. With batch_size=1, this gives an effective batch size of 4 samples.
    val_batches=1,  # Number of validation batches used during each evaluation. Keeping this small makes evaluation faster but also noisier.
    steps_per_report=100, # Print/log training metrics every 200 steps.
    steps_per_eval=200, # Run validation every 400 steps.
    steps_per_save=200, # Save a training checkpoint every 400 steps.
    max_seq_length=1024, # Maximum sequence length used during training. Sequences longer than 1024 tokens are truncated/handled according to the trainer's sequence-processing behavior.
    adapter_file=adapter_file, # File/path where the trained adapter weights are stored.
    grad_checkpoint=True, # Enable gradient checkpointing. Saves memory by recomputing some activations during backward instead of storing all of them during the forward pass. Trade-off: lower memory usage, slightly more computation.
    seq_step_size=None, # Optional sequence chunk/step size. None means no custom sequence stepping/chunking is configured.

    # Quantization-Aware Training (QAT)
    # Disable Quantization-Aware Training for this run.
    # The QAT settings below therefore have no effect unless this is True.
    qat_enable=False,
    qat_bits=quantized_config["bits"], # Target number of bits used by QAT.
    qat_group_size=quantized_config["bits"], # Number of parameters grouped together for quantization.
    qat_mode=quantized_config["mode"], # Quantization mode/strategy used during QAT.
    qat_start_step=1, # Training step at which QAT should begin.
    qat_interval=1, # Apply/update QAT behavior every N training steps.
)

print("\n🚀 Starting ORPO training...")
train_orpo(
    model=model, # Model whose parameters/adapters will be trained.
    args=orpo_config, # ORPO configuration defined above.
    optimizer=opt, # AdamW optimizer and LR schedule.
    train_dataset=CacheDataset(train_set), # Cached training dataset to reduce repeated preprocessing work.
    val_dataset=CacheDataset(eval_set), # Cached validation dataset used during evaluation.
)
print("🎉 SFT training completed!")

## Save merged model

If you have used LoRA. Merge the extra weights learned with LoRA back into the model to obtain a "normal" model checkpoint.

In [ ]:
print("\n🔄 Merging and save LoRA weights...")
save_pretrained_merged(
    model=model, # Trained model.
    tokenizer=tokenizer, # Tokenizer saved alongside the model.
    save_path=new_model_name, # Directory/name for the final merged model.
    de_quantize=True, # Convert quantized weights back to regular weights when saving. You have to turn it to false when qat is enabled.
    remove_adapters=True # Merge/remove adapter structure so the result is a standalone model rather than a base model that requires separate adapter files.
)
print(f"💾 ORPO Merged model saved to: {orpo_config.adapter_file}")